# Day 6 - CLI Personal Finance Tracker
### Python → Gen AI Engineer Journey | Days 1–5 Concepts All in One Project

---

## Concepts Used in This Project

| Day | Topic | Where Used |
|-----|-------|------------|
| Day 1 | Variables, data types, if/else, loops, functions, f-strings | Throughout |
| Day 2 | Lists, Dicts, Sets, String formatting | Transaction storage & display |
| Day 3 | OOP (Classes, `__init__`, dunder methods, dataclasses), generators | `Transaction`, `FinanceTracker` classes |
| Day 4 | Error handling, JSON file I/O, logging, custom exceptions | Save/load data, all user inputs |
| Day 5 | `datetime`, `os`, `pathlib`, async simulation | Timestamps, file paths, async summary |

---

## Project Structure

```
day06_project_finance_tracker/
│
├── 📄 models.py         → Transaction dataclass (Day 3 - OOP)
├── 📄 exceptions.py     → Custom exceptions (Day 4 - Error Handling)
├── 📄 file_utils.py     → JSON save/load with logging (Day 4 - File Handling)
├── 📄 tracker.py        → Core FinanceTracker class (Day 3 - OOP + Day 2 - Data Structures)
├── 📄 async_report.py   → Async summary generator (Day 5 - Async)
└── 📄 main.py           → Main menu loop (Day 1 - Control Flow)
```

In this Colab notebook, all files are written as separate cells so you can understand each part clearly. At the end, everything runs together!


---
## Step 0 - Install & Import Libraries

Yeh libraries hum use karenge. Sab Python ke standard library mein hain - koi extra install nahi chahiye!


In [3]:
# Standard Library - No pip install needed!
import json          # Day 4 - JSON file read/write
import os            # Day 5 - File paths & OS operations
import logging       # Day 4 - Logging instead of print for production code
import asyncio       # Day 5 - Async programming
from datetime import datetime   # Day 5 - Timestamps
from pathlib import Path        # Day 5 - Modern path handling
from dataclasses import dataclass, field, asdict  # Day 3 - Dataclasses
from typing import List, Optional  # Type hints - clean, readable code
from functools import reduce    # Day 3 - Functional programming

print("All libraries imported successfully!")
print(f"Today's date: {datetime.now().strftime('%d %B %Y')}")

All libraries imported successfully!
Today's date: 12 April 2026


---
## Step 1 - Setup: Logging Configuration

**Day 4 Concept: Logging**

> Professionals `print()` nahi karte - woh `logging` use karte hain!
> - `DEBUG` → Detailed info for developers
> - `INFO`  → Normal operations
> - `WARNING` → Something unexpected
> - `ERROR` → Something went wrong


In [4]:
# ============================================================
# LOGGING SETUP - Day 4 Concept
# ============================================================

# basicConfig() ek baar call karo - poore program ke liye kaam karega
logging.basicConfig(
    level=logging.DEBUG,   # Sabse detailed level - sab kuch dikhayega
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)

# Ek named logger banao - professional way
logger = logging.getLogger("FinanceTracker")

# Test karo
logger.info("Logger setup complete ")
logger.debug("Debug mode is ON - will show all messages")

17:04:23 | INFO | Logger setup complete 
17:04:23 | DEBUG | Debug mode is ON - will show all messages


---
## Step 2 - Custom Exceptions

**Day 4 Concept: Custom Exceptions**

> Generic `Exception` use karna beginner level hai.
> Custom exceptions banao - code readable aur debuggable banta hai!


In [6]:
# ============================================================
# FILE: exceptions.py - Day 4 Concept: Custom Exceptions
# ============================================================

class FinanceTrackerError(Exception):
    """
    Base class for all Finance Tracker errors.
    Saari custom exceptions is se inherit karengi.
    
    (Day 3 Concept: Inheritance)
    """
    pass


class InvalidAmountError(FinanceTrackerError):
    """
    Raised jab amount negative ya zero ho.
    Example: add_transaction(-500) → yeh error aayegi
    """
    def __init__(self, amount):
        self.amount = amount
        # super() → parent class ka __init__ call karo (Day 3 - Inheritance)
        super().__init__(f"Amount must be positive! Got: {amount}")


class InvalidCategoryError(FinanceTrackerError):
    """
    Raised jab koi invalid category enter ki jaye.
    """
    def __init__(self, category, valid_categories):
        self.category = category
        super().__init__(
            f"'{category}' is not valid. Choose from: {', '.join(valid_categories)}"
        )


class FileLoadError(FinanceTrackerError):
    """
    Raised jab JSON file load karne mein problem ho.
    """
    pass


# ---- Test karo ----
print("Testing custom exceptions...")
print()

# Test 1: InvalidAmountError
try:
    raise InvalidAmountError(-500)
except InvalidAmountError as e:
    print(f"Caught InvalidAmountError: {e}")

# Test 2: InvalidCategoryError
try:
    raise InvalidCategoryError("gaming", ["food", "rent", "salary"])
except InvalidCategoryError as e:
    print(f"Caught InvalidCategoryError: {e}")

# Test 3: Inheritance check - FinanceTrackerError parent hai
try:
    raise FileLoadError("data.json not found")
except FinanceTrackerError as e:
    print(f"FileLoadError is a FinanceTrackerError: {e}")

print()
print("All exceptions working correctly!")

Testing custom exceptions...

Caught InvalidAmountError: Amount must be positive! Got: -500
Caught InvalidCategoryError: 'gaming' is not valid. Choose from: food, rent, salary
FileLoadError is a FinanceTrackerError: data.json not found

All exceptions working correctly!


---
## Step 3 - Transaction Model (Dataclass)

**Day 3 Concept: Dataclasses & OOP**

> `@dataclass` decorator automatically `__init__`, `__repr__`, `__eq__` banata hai!
> Manually yeh sab likhne ki zarurat nahi.


In [8]:
# ============================================================
# FILE: models.py - Day 3 Concept: OOP + Dataclasses
# ============================================================

@dataclass
class Transaction:
    """
    Ek single financial transaction ko represent karta hai.
    
    @dataclass automatically banata hai:
    - __init__()   → constructor
    - __repr__()   → readable string representation
    - __eq__()     → comparison (== operator)
    
    Day 3 Concepts Used:
    - @dataclass decorator
    - Type hints
    - field() with default_factory
    - Custom method (to_dict, from_dict)
    """
    
    # Required fields - inn ke bina object nahi banega
    transaction_type: str   # "income" ya "expense"
    category: str           # "salary", "food", "rent" etc.
    amount: float           # Kitna paisa
    description: str        # Kya tha yeh transaction
    
    # Optional fields - default value hai
    # field(default_factory=...) → har object ko alag datetime milti hai
    date: str = field(default_factory=lambda: datetime.now().strftime("%Y-%m-%d %H:%M"))
    
    # Unique ID - har transaction ka alag ID
    # datetime se ID banate hain - simple aur guaranteed unique (almost!)
    transaction_id: str = field(
        default_factory=lambda: f"TXN-{datetime.now().strftime('%Y%m%d%H%M%S%f')[:17]}"
    )
    
    def to_dict(self) -> dict:
        """
        Transaction object ko dict mein convert karo.
        JSON save karne ke liye chahiye.
        
        asdict() → dataclass ka saara data dict mein de deta hai
        """
        return asdict(self)
    
    @classmethod
    def from_dict(cls, data: dict) -> 'Transaction':
        """
        Dict se Transaction object banao.
        JSON file se load karne ke liye chahiye.
        
        @classmethod → constructor ki tarah kaam karta hai
        cls → class khud (Transaction)
        """
        return cls(**data)  # ** → dict ko keyword arguments mein unpack karo (Day 3 - **kwargs)
    
    def __str__(self) -> str:
        """
        Dunder method - print() karne par yeh string dikhegi.
        Day 3 Concept: Dunder Methods
        """
        emoji = " " if self.transaction_type == "income" else " "
        sign = "+" if self.transaction_type == "income" else "-"
        return (
            f"{emoji} [{self.date}] {self.category.upper():12} "
            f"{sign}Rs.{self.amount:>10.2f}  |  {self.description}"
        )


# ---- Test karo ----
print("Testing Transaction dataclass...")
print()

# Object banao
t1 = Transaction(
    transaction_type="income",
    category="salary",
    amount=50000.0,
    description="Monthly salary from company"
)

t2 = Transaction(
    transaction_type="expense",
    category="food",
    amount=1500.0,
    description="Lunch at restaurant"
)

# __str__ test (dunder method)
print("Transaction objects:")
print(t1)
print(t2)
print()

# to_dict test
print("to_dict() output:")
print(json.dumps(t1.to_dict(), indent=2))
print()

# from_dict test (round-trip)
t1_dict = t1.to_dict()
t1_restored = Transaction.from_dict(t1_dict)
print(f"from_dict() round-trip works: {t1 == t1_restored}")

Testing Transaction dataclass...

Transaction objects:
  [2026-04-12 17:06] SALARY       +Rs.  50000.00  |  Monthly salary from company
  [2026-04-12 17:06] FOOD         -Rs.   1500.00  |  Lunch at restaurant

to_dict() output:
{
  "transaction_type": "income",
  "category": "salary",
  "amount": 50000.0,
  "description": "Monthly salary from company",
  "date": "2026-04-12 17:06",
  "transaction_id": "TXN-20260412170615725"
}

from_dict() round-trip works: True


---
## Step 4 - File Utilities (JSON Save/Load)

**Day 4 Concept: File Handling + Error Handling + Logging**

> Yeh woh layer hai jo data ko disk pe save karta hai.
> Production code mein always proper error handling hota hai!


In [10]:
# ============================================================
# FILE: file_utils.py - Day 4: File Handling, Error Handling, Logging
# ============================================================

# File kahan save hogi - pathlib ka use (Day 5)
DATA_FILE = Path("finance_data.json")


def save_transactions(transactions: List[Transaction]) -> bool:
    """
    Transactions list ko JSON file mein save karo.
    
    Day 4 Concepts:
    - JSON file write
    - try/except/finally
    - Logging
    
    Returns: True if saved successfully, False otherwise
    """
    try:
        logger.info(f"Saving {len(transactions)} transactions to {DATA_FILE}...")
        
        # List comprehension → har Transaction ko dict mein convert karo (Day 2 + Day 3)
        data = {
            "last_saved": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "total_transactions": len(transactions),
            "transactions": [t.to_dict() for t in transactions]  # List comprehension!
        }
        
        # JSON file write karo
        with open(DATA_FILE, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        logger.info(f"Data saved successfully to '{DATA_FILE}'")
        return True
        
    except PermissionError:
        # File access nahi mila
        logger.error(f"Permission denied: Cannot write to '{DATA_FILE}'")
        return False
        
    except Exception as e:
        # Koi bhi unexpected error
        logger.error(f"Unexpected error while saving: {e}")
        return False


def load_transactions() -> List[Transaction]:
    """
    JSON file se transactions load karo.
    
    Day 4 Concepts:
    - JSON file read
    - try/except with multiple except blocks
    - FileNotFoundError handling
    
    Returns: List of Transaction objects
    """
    # Pehle check karo file exist karti hai ya nahi (Day 5 - pathlib)
    if not DATA_FILE.exists():
        logger.info(f"No existing data file found. Starting fresh.")
        return []  # Empty list return karo
    
    try:
        logger.info(f"Loading transactions from '{DATA_FILE}'...")
        
        with open(DATA_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        # Dict list ko Transaction objects mein convert karo
        transactions = [
            Transaction.from_dict(t) 
            for t in data.get("transactions", [])  # .get() → KeyError se bachao
        ]
        
        logger.info(f"Loaded {len(transactions)} transactions successfully")
        return transactions
        
    except json.JSONDecodeError as e:
        # File corrupt ho gayi
        logger.error(f"File is corrupted (invalid JSON): {e}")
        raise FileLoadError(f"Could not parse '{DATA_FILE}'. File may be corrupted.") from e
        
    except KeyError as e:
        # JSON mein expected key nahi mili
        logger.error(f"Missing key in data: {e}")
        return []
        
    except Exception as e:
        logger.error(f"Unexpected error while loading: {e}")
        return []


def get_file_info() -> dict:
    """
    File ke baare mein info return karo.
    Day 5 - pathlib + os
    """
    if not DATA_FILE.exists():
        return {"exists": False}
    
    stat = DATA_FILE.stat()
    return {
        "exists": True,
        "path": str(DATA_FILE.absolute()),
        "size_kb": round(stat.st_size / 1024, 2),
        "last_modified": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S")
    }


# ---- Test karo ----
print("Testing file utilities...")
print()

# Test transactions
test_transactions = [
    Transaction("income", "salary", 50000.0, "Monthly salary"),
    Transaction("expense", "food", 2000.0, "Grocery shopping"),
    Transaction("expense", "rent", 15000.0, "House rent"),
]

# Save test
saved = save_transactions(test_transactions)
print(f"Save result: {saved}")
print()

# Load test
loaded = load_transactions()
print(f"Loaded {len(loaded)} transactions")
for t in loaded:
    print(f"  {t}")
print()

# File info
info = get_file_info()
print(f"File info: {info}")

17:06:56 | INFO | Saving 3 transactions to finance_data.json...
17:06:56 | INFO | Data saved successfully to 'finance_data.json'
17:06:56 | INFO | Loading transactions from 'finance_data.json'...
17:06:56 | INFO | Loaded 3 transactions successfully


Testing file utilities...

Save result: True

Loaded 3 transactions
    [2026-04-12 17:06] SALARY       +Rs.  50000.00  |  Monthly salary
    [2026-04-12 17:06] FOOD         -Rs.   2000.00  |  Grocery shopping
    [2026-04-12 17:06] RENT         -Rs.  15000.00  |  House rent

File info: {'exists': True, 'path': 'c:\\Users\\shaur\\Downloads\\finance_data.json', 'size_kb': 0.77, 'last_modified': '2026-04-12 17:06:56'}


---
## Step 5 - Core Finance Tracker Class

**Day 2 + Day 3 Concept: OOP + Data Structures**

> Yeh main class hai jo saari business logic handle karta hai.
> Real apps mein yahi pattern follow hota hai!


In [11]:
# ============================================================
# FILE: tracker.py - Day 2 (Data Structures) + Day 3 (OOP)
# ============================================================

class FinanceTracker:
    """
    Personal Finance Tracker - Main Class.
    
    Day 3 Concepts Used:
    - Class variables vs instance variables
    - __init__ constructor
    - Dunder methods (__str__, __len__, __iter__)
    - Properties (@property)
    - Generator methods (yield)
    - Encapsulation (private methods with _)
    
    Day 2 Concepts Used:
    - Lists → transactions store karne ke liye
    - Dicts → category-wise summary ke liye
    - Sets → unique categories nikalne ke liye
    """
    
    # Class variable - sabhi objects ke liye same (Day 3)
    VALID_INCOME_CATEGORIES = {"salary", "freelance", "investment", "gift", "other_income"}
    VALID_EXPENSE_CATEGORIES = {"food", "rent", "transport", "entertainment", "health", 
                                 "education", "shopping", "utilities", "other_expense"}
    
    def __init__(self, user_name: str = "User"):
        """
        Constructor - object banate waqt call hota hai.
        Day 3 Concept: __init__
        """
        # Instance variables - har object ka apna data
        self.user_name = user_name
        self._transactions: List[Transaction] = []  # Private variable (convention: _ prefix)
        self._created_at = datetime.now()
        
        # File se data load karo (Day 4 - File Handling)
        self._load_data()
        logger.info(f"FinanceTracker initialized for user: {user_name}")
    
    # ==================== DUNDER METHODS (Day 3) ====================
    
    def __str__(self) -> str:
        """print(tracker) karne par yeh dikhega"""
        return f"FinanceTracker(user='{self.user_name}', transactions={len(self._transactions)})"
    
    def __repr__(self) -> str:
        """Developer ke liye detailed representation"""
        return f"FinanceTracker(user='{self.user_name}', balance=Rs.{self.balance:.2f})"
    
    def __len__(self) -> int:
        """len(tracker) → total transactions count"""
        return len(self._transactions)
    
    def __iter__(self):
        """for t in tracker → transactions pe iterate kar sako"""
        return iter(self._transactions)
    
    # ==================== PROPERTIES (Day 3 - Encapsulation) ====================
    
    @property
    def balance(self) -> float:
        """
        Current balance calculate karo.
        @property → method ko attribute ki tarah access kar sako: tracker.balance
        
        Day 3 - map() + filter() + reduce()
        """
        incomes = sum(t.amount for t in self._transactions if t.transaction_type == "income")
        expenses = sum(t.amount for t in self._transactions if t.transaction_type == "expense")
        return incomes - expenses
    
    @property
    def total_income(self) -> float:
        """Total income calculate karo - generator expression use karo (memory efficient!)"""
        return sum(t.amount for t in self._transactions if t.transaction_type == "income")
    
    @property
    def total_expenses(self) -> float:
        """Total expenses calculate karo"""
        return sum(t.amount for t in self._transactions if t.transaction_type == "expense")
    
    @property
    def categories_used(self) -> set:
        """Unique categories - Set use karo (Day 2 - Sets)"""
        return {t.category for t in self._transactions}  # Set comprehension!
    
    # ==================== MAIN METHODS ====================
    
    def add_transaction(
        self,
        transaction_type: str,
        category: str,
        amount: float,
        description: str
    ) -> Transaction:
        """
        Nayi transaction add karo.
        
        Day 4 - Custom exceptions + validation
        Day 3 - OOP
        """
        # Validation - Day 4: Custom Exceptions
        self._validate_input(transaction_type, category, amount)
        
        # Transaction object banao (Day 3 - OOP)
        transaction = Transaction(
            transaction_type=transaction_type,
            category=category,
            amount=float(amount),
            description=description
        )
        
        # List mein add karo (Day 2 - Lists)
        self._transactions.append(transaction)
        
        # File mein save karo (Day 4 - File Handling)
        self._save_data()
        
        logger.info(f"Transaction added: {transaction.transaction_id}")
        return transaction
    
    def get_transactions_by_category(self, category: str) -> List[Transaction]:
        """
        Category se transactions filter karo.
        Day 2 + Day 3: filter() + List Comprehension
        """
        # List comprehension → filter karo (Day 1 + Day 2)
        return [t for t in self._transactions if t.category == category]
    
    def get_transactions_by_type(self, t_type: str) -> List[Transaction]:
        """Income ya expense ke hisaab se filter karo"""
        return [t for t in self._transactions if t.transaction_type == t_type]
    
    def get_category_summary(self) -> dict:
        """
        Category-wise spending summary.
        Day 2 - Dictionaries
        Day 3 - Generators
        """
        summary = {}  # Empty dict
        
        for transaction in self._transactions:  # Iterate over transactions
            cat = transaction.category
            
            # Dict mein key hai to update karo, nahi to add karo
            if cat not in summary:
                summary[cat] = {"total": 0.0, "count": 0, "type": transaction.transaction_type}
            
            summary[cat]["total"] += transaction.amount
            summary[cat]["count"] += 1
        
        return summary
    
    def transaction_generator(self):
        """
        Generator function - ek ek transaction yield karo.
        Day 3 Concept: Generators (memory efficient!)
        
        Normal list return karne ki jagah generator use karo
        kyunki agar 1 million transactions hoon toh list memory mein nahi jayegi.
        """
        for transaction in self._transactions:
            yield transaction  # yield → ek ek element return karo
    
    def delete_transaction(self, transaction_id: str) -> bool:
        """
        ID se transaction delete karo.
        Day 2 - List operations
        """
        original_count = len(self._transactions)
        
        # List comprehension se filter out karo (delete ki jagah filter use karo)
        self._transactions = [
            t for t in self._transactions 
            if t.transaction_id != transaction_id
        ]
        
        if len(self._transactions) < original_count:
            self._save_data()
            logger.info(f"Transaction deleted: {transaction_id}")
            return True
        else:
            logger.warning(f"Transaction not found: {transaction_id}")
            return False
    
    # ==================== PRIVATE METHODS (Day 3 - Encapsulation) ====================
    
    def _validate_input(self, transaction_type: str, category: str, amount: float):
        """
        Private method - sirf class ke andar use hoga.
        _ prefix = private (convention, not enforced by Python).
        
        Day 4 - Custom Exceptions
        """
        # Amount validation
        if amount <= 0:
            raise InvalidAmountError(amount)
        
        # Type validation
        if transaction_type not in ("income", "expense"):
            raise FinanceTrackerError(f"type must be 'income' or 'expense', got '{transaction_type}'")
        
        # Category validation
        valid = (
            self.VALID_INCOME_CATEGORIES 
            if transaction_type == "income" 
            else self.VALID_EXPENSE_CATEGORIES
        )
        if category not in valid:
            raise InvalidCategoryError(category, sorted(valid))
    
    def _save_data(self):
        """Internal save - file_utils.py ka use karo"""
        save_transactions(self._transactions)
    
    def _load_data(self):
        """Internal load - file_utils.py ka use karo"""
        try:
            self._transactions = load_transactions()
        except FileLoadError as e:
            logger.error(f"Could not load data: {e}. Starting with empty tracker.")
            self._transactions = []


print("FinanceTracker class defined successfully!")
print()
print("Class features:")
print("  - add_transaction()          → nayi transaction add karo")
print("  - get_transactions_by_type() → income/expense filter")
print("  - get_category_summary()     → category-wise breakdown")
print("  - transaction_generator()    → memory-efficient iteration")
print("  - delete_transaction()       → ID se delete karo")
print("  - balance, total_income...   → properties (auto-calculated)")

FinanceTracker class defined successfully!

Class features:
  - add_transaction()          → nayi transaction add karo
  - get_transactions_by_type() → income/expense filter
  - get_category_summary()     → category-wise breakdown
  - transaction_generator()    → memory-efficient iteration
  - delete_transaction()       → ID se delete karo
  - balance, total_income...   → properties (auto-calculated)


---
## Step 6 - Async Report Generator

**Day 5 Concept: Async/Await + asyncio**

> Real Gen AI apps mein API calls async hote hain.
> Yahan hum fake async tasks simulate kar rahe hain - same pattern use hota hai!


In [12]:
# ============================================================
# FILE: async_report.py - Day 5: Async/Await + asyncio
# ============================================================

async def fetch_spending_analysis(category: str, amount: float) -> dict:
    """
    Fake async function - jaise real API call hoti hai.
    
    Real world mein yahan:
    - AI API call hoga (OpenAI, Anthropic)
    - Database query hogi
    - External service call hogi
    
    Yahan hum asyncio.sleep se simulate kar rahe hain.
    """
    # Simulate network delay (0.1 second)
    await asyncio.sleep(0.1)
    
    # Simple spending advice (Day 1 - if/else, Day 2 - dict)
    advice_map = {
        "food": "Consider meal prepping to reduce food expenses by 20-30%!",
        "entertainment": "Entertainment spending is high. Try free activities!",
        "transport": "Consider carpooling or public transport to save money.",
        "shopping": "Review subscriptions and one-time purchases carefully.",
        "health": "Health investment is important - keep it up!",
        "education": "Learning is investing in yourself! Great choice.",
        "rent": "Rent is a fixed cost. Focus on reducing variable expenses.",
        "utilities": "Check for energy-saving options to reduce utility bills.",
    }
    
    return {
        "category": category,
        "amount": amount,
        "advice": advice_map.get(category, "Track this category closely.")
    }


async def generate_full_report(tracker: FinanceTracker) -> dict:
    """
    Poori report generate karo - multiple async tasks parallel mein!
    
    asyncio.gather() → saari tasks ek saath chalao (parallel!)
    Normal code mein yeh ek ke baad ek chalte - async mein sab ek saath!
    """
    logger.info("Starting async report generation...")
    
    # Category summary nikalo (Day 2 - Dict)
    category_summary = tracker.get_category_summary()
    
    if not category_summary:
        return {"error": "No transactions found"}
    
    # Saari categories ke liye async tasks banao
    # asyncio.gather() → sab parallel mein chalao!
    tasks = [
        fetch_spending_analysis(cat, data["total"])
        for cat, data in category_summary.items()
        if data["type"] == "expense"  # Sirf expenses ke liye advice
    ]
    
    if tasks:
        # await → sab tasks complete hone tak ruko
        analyses = await asyncio.gather(*tasks)
    else:
        analyses = []
    
    report = {
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "user": tracker.user_name,
        "summary": {
            "total_income": tracker.total_income,
            "total_expenses": tracker.total_expenses,
            "balance": tracker.balance,
            "total_transactions": len(tracker),
            "categories_used": list(tracker.categories_used)
        },
        "spending_advice": list(analyses)
    }
    
    logger.info("Async report generation complete!")
    return report


print("Async report functions defined!")
print("asyncio.gather() → sab tasks parallel mein chalti hain - real Gen AI pattern!")

Async report functions defined!
asyncio.gather() → sab tasks parallel mein chalti hain - real Gen AI pattern!


---
## Step 7 - Display Helpers

**Day 1 Concept: Functions + F-strings**

> Clean output ke liye helper functions - code reusability!


In [13]:
# ============================================================
# Display Helper Functions - Day 1: Functions, F-strings, Loops
# ============================================================

def print_header(title: str, width: int = 60):
    """Sundar header print karo - Day 1: String formatting"""
    print("\n" + "=" * width)
    print(f"  {title}")
    print("=" * width)


def print_divider(char: str = "-", width: int = 60):
    """Divider line print karo"""
    print(char * width)


def display_balance(tracker: FinanceTracker):
    """
    Balance summary dikhao.
    Day 1: if/else + F-strings
    Day 3: @property use karo
    """
    print_header("BALANCE SUMMARY")
    print(f"User          : {tracker.user_name}")
    print(f"Total Income  : Rs. {tracker.total_income:>12,.2f}")
    print(f"Total Expenses: Rs. {tracker.total_expenses:>12,.2f}")
    print_divider()
    
    # Day 1 - if/else
    balance = tracker.balance
    emoji = "🟢" if balance >= 0 else "🔴"
    status = "SURPLUS" if balance >= 0 else "DEFICIT"
    print(f"  {emoji} Balance ({status}): Rs. {balance:>12,.2f}")
    print_divider()


def display_all_transactions(tracker: FinanceTracker):
    """
    Saari transactions dikhao.
    Day 1: Loops
    Day 3: __iter__ dunder method use ho raha hai
    Day 3: Generator pattern
    """
    print_header(f"ALL TRANSACTIONS ({len(tracker)} total)")
    
    if len(tracker) == 0:
        print("  No transactions yet. Add some!")
        return
    
    # Generator use karo - Day 3!
    for i, transaction in enumerate(tracker.transaction_generator(), 1):
        print(f"  {i:>3}. {transaction}")  # __str__ dunder method call ho raha hai
    
    print_divider()


def display_category_summary(tracker: FinanceTracker):
    """
    Category-wise summary dikhao.
    Day 2 - Dict
    Day 1 - Loops, F-strings
    """
    print_header("CATEGORY-WISE SUMMARY")
    
    summary = tracker.get_category_summary()
    
    if not summary:
        print("  No data yet.")
        return
    
    # Sort by amount (highest first) - Day 2: sorted() with key
    sorted_summary = sorted(summary.items(), key=lambda x: x[1]["total"], reverse=True)
    
    print(f"  {'Category':<20} {'Type':<10} {'Amount':>12} {'Count':>6}")
    print_divider()
    
    for category, data in sorted_summary:
        emoji = "📈" if data["type"] == "income" else "📉"
        print(f"  {emoji} {category:<18} {data['type']:<10} Rs. {data['total']:>10,.2f} {data['count']:>5}x")
    
    print_divider()


def display_report(report: dict):
    """Async report dikhao"""
    print_header("AI-POWERED SPENDING ANALYSIS REPORT")
    print(f"  Generated: {report.get('generated_at')}")
    print_divider()
    
    summary = report.get("summary", {})
    print(f"  Income   : Rs. {summary.get('total_income', 0):,.2f}")
    print(f"  Expenses : Rs. {summary.get('total_expenses', 0):,.2f}")
    print(f"  Balance  : Rs. {summary.get('balance', 0):,.2f}")
    print()
    
    advice_list = report.get("spending_advice", [])
    if advice_list:
        print("SPENDING ADVICE:")
        for advice in advice_list:
            print(f"  • [{advice['category'].upper()}] Rs.{advice['amount']:,.2f}")
            print(f"    → {advice['advice']}")
    
    print_divider()


print("Display helper functions ready!")

Display helper functions ready!


---
## Step 8 - Main Application (Everything Together!)

**Ab saare concepts ek jagah use hote hain!**

> Yeh interactive menu hai - real CLI app ki tarah kaam karta hai!


In [14]:
# ============================================================
# FILE: main.py - Day 1: Control Flow, Input/Output, Functions
# ============================================================

def get_float_input(prompt: str) -> float:
    """
    User se valid float input lo.
    Day 4 - try/except
    Day 1 - while loop
    """
    while True:  # Day 1 - Loop
        try:
            value = float(input(prompt))  # Day 1 - input()
            if value <= 0:
                print("Amount must be positive! Try again.")
                continue
            return value
        except ValueError:  # Day 4 - Exception handling
            print("Invalid amount. Enter a number like 500 or 1500.5")


def get_choice_input(prompt: str, valid_choices: list) -> str:
    """
    User se valid choice lo.
    Day 1 - while loop, if/else
    Day 2 - List membership check
    """
    while True:
        choice = input(prompt).strip().lower()
        if choice in valid_choices:
            return choice
        print(f"Invalid choice. Please enter one of: {', '.join(valid_choices)}")


def show_category_menu(transaction_type: str) -> str:
    """
    Type ke hisaab se valid categories dikhao.
    Day 1 - if/else
    Day 2 - List
    """
    if transaction_type == "income":
        categories = sorted(FinanceTracker.VALID_INCOME_CATEGORIES)  # Class variable access
    else:
        categories = sorted(FinanceTracker.VALID_EXPENSE_CATEGORIES)
    
    print("\nAvailable categories:")
    for i, cat in enumerate(categories, 1):  # enumerate - Day 3
        print(f"     {i}. {cat}")
    
    return get_choice_input("Enter category: ", categories)


def handle_add_transaction(tracker: FinanceTracker):
    """
    Transaction add karne ka complete flow.
    Day 4 - try/except with custom exceptions
    """
    print_header("ADD TRANSACTION")
    
    try:
        # Step 1: Type choose karo
        t_type = get_choice_input(
            "Type (income/expense): ",
            ["income", "expense"]
        )
        
        # Step 2: Category choose karo
        category = show_category_menu(t_type)
        
        # Step 3: Amount enter karo
        amount = get_float_input("Amount (Rs.): ")
        
        # Step 4: Description enter karo
        description = input("Description: ").strip()
        if not description:
            description = f"{t_type.capitalize()} - {category}"
        
        # Transaction add karo
        transaction = tracker.add_transaction(t_type, category, amount, description)
        
        print(f"\nTransaction added successfully!")
        print(f"  {transaction}")
        print(f"New Balance: Rs. {tracker.balance:,.2f}")
        
    except InvalidAmountError as e:
        print(f"\n  ❌ {e}")
    except InvalidCategoryError as e:
        print(f"\n  ❌ {e}")
    except FinanceTrackerError as e:
        print(f"\n  ❌ Tracker Error: {e}")
    except KeyboardInterrupt:
        print("\n  ↩️  Cancelled.")


def handle_view_transactions(tracker: FinanceTracker):
    """Transactions view karne ke options"""
    print_header("VIEW TRANSACTIONS")
    print("  1. All transactions")
    print("  2. Income only")
    print("  3. Expenses only")
    print("  4. By category")
    print("  5. Back")
    
    choice = get_choice_input("Choose: ", ["1", "2", "3", "4", "5"])
    
    if choice == "1":
        display_all_transactions(tracker)
    elif choice == "2":
        incomes = tracker.get_transactions_by_type("income")
        print_header(f"INCOME TRANSACTIONS ({len(incomes)})")
        for i, t in enumerate(incomes, 1):
            print(f"  {i}. {t}")
    elif choice == "3":
        expenses = tracker.get_transactions_by_type("expense")
        print_header(f"EXPENSE TRANSACTIONS ({len(expenses)})")
        for i, t in enumerate(expenses, 1):
            print(f"  {i}. {t}")
    elif choice == "4":
        cat = input("Category name: ").strip().lower()
        results = tracker.get_transactions_by_category(cat)
        print_header(f"CATEGORY: {cat.upper()} ({len(results)} found)")
        for i, t in enumerate(results, 1):
            print(f"  {i}. {t}")


async def handle_report(tracker: FinanceTracker):
    """Async report generate karo"""
    print_header("GENERATING ASYNC REPORT...")
    print("  (Simulating AI analysis - async tasks running in parallel...)")
    
    report = await generate_full_report(tracker)
    display_report(report)


print("Main application functions ready!")

Main application functions ready!


---
## Step 9 - DEMO: Sample Data + Full Walkthrough

Pehle ek demo run karte hain jismein automatically sample data add hoga aur saare features dikhenge!


In [15]:
# ============================================================
# DEMO - Saare concepts ek saath dekho!
# ============================================================

import os

# Clean slate ke liye purana data file delete karo
if os.path.exists("finance_data.json"):
    os.remove("finance_data.json")
    print("Old data cleared for fresh demo")

print()
print("=" * 60)
print("PERSONAL FINANCE TRACKER - DEMO MODE")
print("Python → Gen AI Engineer Journey | Day 6 Project")
print("=" * 60)

# ---- Step 1: Tracker banao (Day 3 - OOP) ----
print("\nStep 1: Creating FinanceTracker object (Day 3 - OOP)")
tracker = FinanceTracker(user_name="Rahul Sharma")
print(f"Tracker created: {tracker}")  # __str__ dunder method
print(f"repr: {repr(tracker)}")       # __repr__ dunder method

# ---- Step 2: Transactions add karo (Day 1-4) ----
print("\nStep 2: Adding sample transactions...")

sample_data = [
    # (type, category, amount, description)
    ("income",  "salary",        50000.00, "Monthly salary - TCS"),
    ("income",  "freelance",      8000.00, "Website design project"),
    ("expense", "rent",          15000.00, "House rent - January"),
    ("expense", "food",           4500.00, "Monthly groceries"),
    ("expense", "food",           2200.00, "Restaurant bills"),
    ("expense", "transport",      1800.00, "Petrol + Ola cabs"),
    ("expense", "education",      2999.00, "Udemy Python course"),
    ("expense", "entertainment",  1500.00, "Netflix + movies"),
    ("expense", "health",         3500.00, "Gym membership"),
    ("expense", "utilities",      2100.00, "Electricity + internet"),
    ("income",  "investment",     3000.00, "Stock dividends"),
    ("expense", "shopping",       4200.00, "New clothes + shoes"),
]

# Saara data add karo - try/except (Day 4)
added = 0
for t_type, category, amount, desc in sample_data:
    try:
        tracker.add_transaction(t_type, category, amount, desc)
        added += 1
    except FinanceTrackerError as e:
        print(f"Error: {e}")

print(f"  ✅ Added {added}/{len(sample_data)} transactions")
print(f"  Total in tracker (len): {len(tracker)}")  # __len__ dunder method

17:13:37 | INFO | No existing data file found. Starting fresh.
17:13:37 | INFO | FinanceTracker initialized for user: Rahul Sharma
17:13:37 | INFO | Saving 1 transactions to finance_data.json...
17:13:37 | INFO | Data saved successfully to 'finance_data.json'
17:13:37 | INFO | Transaction added: TXN-20260412171337754
17:13:37 | INFO | Saving 2 transactions to finance_data.json...
17:13:37 | INFO | Data saved successfully to 'finance_data.json'
17:13:37 | INFO | Transaction added: TXN-20260412171337759
17:13:37 | INFO | Saving 3 transactions to finance_data.json...
17:13:37 | INFO | Data saved successfully to 'finance_data.json'
17:13:37 | INFO | Transaction added: TXN-20260412171337764
17:13:37 | INFO | Saving 4 transactions to finance_data.json...
17:13:37 | INFO | Data saved successfully to 'finance_data.json'
17:13:37 | INFO | Transaction added: TXN-20260412171337767
17:13:37 | INFO | Saving 5 transactions to finance_data.json...
17:13:37 | INFO | Data saved successfully to 'finance

Old data cleared for fresh demo

PERSONAL FINANCE TRACKER - DEMO MODE
Python → Gen AI Engineer Journey | Day 6 Project

Step 1: Creating FinanceTracker object (Day 3 - OOP)
Tracker created: FinanceTracker(user='Rahul Sharma', transactions=0)
repr: FinanceTracker(user='Rahul Sharma', balance=Rs.0.00)

Step 2: Adding sample transactions...
  ✅ Added 12/12 transactions
  Total in tracker (len): 12


In [17]:
# ---- Step 3: Balance dikhao ----
print("Step 3: Balance Summary (Day 3 - @property)")
display_balance(tracker)

Step 3: Balance Summary (Day 3 - @property)

  BALANCE SUMMARY
User          : Rahul Sharma
Total Income  : Rs.    61,000.00
Total Expenses: Rs.    37,799.00
------------------------------------------------------------
  🟢 Balance (SURPLUS): Rs.    23,201.00
------------------------------------------------------------


In [19]:
# ---- Step 4: Saari transactions dikhao ----
print("Step 4: All Transactions (Day 3 - Generator + Dunder Methods)")
display_all_transactions(tracker)

Step 4: All Transactions (Day 3 - Generator + Dunder Methods)

  ALL TRANSACTIONS (12 total)
    1.   [2026-04-12 17:13] SALARY       +Rs.  50000.00  |  Monthly salary - TCS
    2.   [2026-04-12 17:13] FREELANCE    +Rs.   8000.00  |  Website design project
    3.   [2026-04-12 17:13] RENT         -Rs.  15000.00  |  House rent - January
    4.   [2026-04-12 17:13] FOOD         -Rs.   4500.00  |  Monthly groceries
    5.   [2026-04-12 17:13] FOOD         -Rs.   2200.00  |  Restaurant bills
    6.   [2026-04-12 17:13] TRANSPORT    -Rs.   1800.00  |  Petrol + Ola cabs
    7.   [2026-04-12 17:13] EDUCATION    -Rs.   2999.00  |  Udemy Python course
    8.   [2026-04-12 17:13] ENTERTAINMENT -Rs.   1500.00  |  Netflix + movies
    9.   [2026-04-12 17:13] HEALTH       -Rs.   3500.00  |  Gym membership
   10.   [2026-04-12 17:13] UTILITIES    -Rs.   2100.00  |  Electricity + internet
   11.   [2026-04-12 17:13] INVESTMENT   +Rs.   3000.00  |  Stock dividends
   12.   [2026-04-12 17:13] SHOPPING 

In [21]:
# ---- Step 5: Category summary ----
print("Step 5: Category Summary (Day 2 - Dicts + Sets)")
display_category_summary(tracker)

# Set demonstration - Day 2
print(f"\nUnique categories used (Set): {tracker.categories_used}")

Step 5: Category Summary (Day 2 - Dicts + Sets)

  CATEGORY-WISE SUMMARY
  Category             Type             Amount  Count
------------------------------------------------------------
  📈 salary             income     Rs.  50,000.00     1x
  📉 rent               expense    Rs.  15,000.00     1x
  📈 freelance          income     Rs.   8,000.00     1x
  📉 food               expense    Rs.   6,700.00     2x
  📉 shopping           expense    Rs.   4,200.00     1x
  📉 health             expense    Rs.   3,500.00     1x
  📈 investment         income     Rs.   3,000.00     1x
  📉 education          expense    Rs.   2,999.00     1x
  📉 utilities          expense    Rs.   2,100.00     1x
  📉 transport          expense    Rs.   1,800.00     1x
  📉 entertainment      expense    Rs.   1,500.00     1x
------------------------------------------------------------

Unique categories used (Set): {'entertainment', 'rent', 'freelance', 'salary', 'health', 'food', 'investment', 'utilities', 'shopping'

In [22]:
# ---- Step 6: Filter karo ----
print("Step 6: Filtering Transactions (Day 2 - List Comprehension)")

# Income transactions
incomes = tracker.get_transactions_by_type("income")
print_header(f"INCOME ONLY ({len(incomes)} transactions)")
for i, t in enumerate(incomes, 1):
    print(f"  {i}. {t}")

print()

# Food category
food_txns = tracker.get_transactions_by_category("food")
print_header(f"FOOD CATEGORY ({len(food_txns)} transactions)")
for i, t in enumerate(food_txns, 1):
    print(f"  {i}. {t}")
total_food = sum(t.amount for t in food_txns)  # Generator expression!
print(f"  Total food spending: Rs. {total_food:,.2f}")

Step 6: Filtering Transactions (Day 2 - List Comprehension)

  INCOME ONLY (3 transactions)
  1.   [2026-04-12 17:13] SALARY       +Rs.  50000.00  |  Monthly salary - TCS
  2.   [2026-04-12 17:13] FREELANCE    +Rs.   8000.00  |  Website design project
  3.   [2026-04-12 17:13] INVESTMENT   +Rs.   3000.00  |  Stock dividends


  FOOD CATEGORY (2 transactions)
  1.   [2026-04-12 17:13] FOOD         -Rs.   4500.00  |  Monthly groceries
  2.   [2026-04-12 17:13] FOOD         -Rs.   2200.00  |  Restaurant bills
  Total food spending: Rs. 6,700.00


In [23]:
# ---- Step 7: Generator demo ----
print("Step 7: Generator Pattern (Day 3 - Generators)")
print()
print("  Normal list: sabhi items ek saath memory mein")
print("  Generator:   ek ek item - memory efficient!")
print()

# Generator use karo - Day 3
gen = tracker.transaction_generator()
print(f"  Generator object: {gen}")
print("  First 3 via generator (next()):")
for _ in range(3):
    t = next(gen)  # next() → ek item lao
    print(f"    → {t}")
print("  (Remaining items still in generator - not loaded yet!)")

Step 7: Generator Pattern (Day 3 - Generators)

  Normal list: sabhi items ek saath memory mein
  Generator:   ek ek item - memory efficient!

  Generator object: <generator object FinanceTracker.transaction_generator at 0x0000029678CDC1C0>
  First 3 via generator (next()):
    →   [2026-04-12 17:13] SALARY       +Rs.  50000.00  |  Monthly salary - TCS
    →   [2026-04-12 17:13] FREELANCE    +Rs.   8000.00  |  Website design project
    →   [2026-04-12 17:13] RENT         -Rs.  15000.00  |  House rent - January
  (Remaining items still in generator - not loaded yet!)


In [24]:
# ---- Step 8: Error handling demo ----
print("Step 8: Error Handling (Day 4 - Custom Exceptions)")
print()

error_tests = [
    # (type, category, amount, description, expected_error)
    ("expense", "food", -500, "Negative amount", "InvalidAmountError"),
    ("expense", "gaming", 1000, "Invalid category", "InvalidCategoryError"),
    ("spending", "food", 100, "Invalid type", "FinanceTrackerError"),
]

for t_type, category, amount, desc, expected in error_tests:
    try:
        tracker.add_transaction(t_type, category, amount, desc)
    except InvalidAmountError as e:
        print(f"  ✅ Caught {expected}: {e}")
    except InvalidCategoryError as e:
        print(f"  ✅ Caught {expected}: {e}")
    except FinanceTrackerError as e:
        print(f"  ✅ Caught {expected}: {e}")

Step 8: Error Handling (Day 4 - Custom Exceptions)

  ✅ Caught InvalidAmountError: Amount must be positive! Got: -500
  ✅ Caught InvalidCategoryError: 'gaming' is not valid. Choose from: education, entertainment, food, health, other_expense, rent, shopping, transport, utilities
  ✅ Caught FinanceTrackerError: type must be 'income' or 'expense', got 'spending'


In [25]:
# ---- Step 9: File handling demo ----
print("Step 9: File Handling (Day 4 - JSON + pathlib)")
print()

file_info = get_file_info()
print(f"File path   : {file_info['path']}")
print(f"File size   : {file_info['size_kb']} KB")
print(f"Last saved  : {file_info['last_modified']}")
print()

# JSON content dikhao (first 20 lines)
with open("finance_data.json", "r") as f:
    content = f.read()
lines = content.split("\n")
print("  JSON file preview (first 20 lines):")
for line in lines[:20]:
    print(f"    {line}")
print(f"  ... ({len(lines)} total lines)")

Step 9: File Handling (Day 4 - JSON + pathlib)

File path   : c:\Users\shaur\Downloads\finance_data.json
File size   : 2.87 KB
Last saved  : 2026-04-12 17:13:37

  JSON file preview (first 20 lines):
    {
      "last_saved": "2026-04-12 17:13:37",
      "total_transactions": 12,
      "transactions": [
        {
          "transaction_type": "income",
          "category": "salary",
          "amount": 50000.0,
          "description": "Monthly salary - TCS",
          "date": "2026-04-12 17:13",
          "transaction_id": "TXN-20260412171337754"
        },
        {
          "transaction_type": "income",
          "category": "freelance",
          "amount": 8000.0,
          "description": "Website design project",
          "date": "2026-04-12 17:13",
          "transaction_id": "TXN-20260412171337759"
        },
  ... (102 total lines)


In [26]:
# ---- Step 10: Async report ----
print("Step 10: Async Report (Day 5 - asyncio)")
print()

# Colab mein await direct use kar sakte hain (Jupyter magic!)
report = await generate_full_report(tracker)
display_report(report)

17:15:47 | INFO | Starting async report generation...
17:15:48 | INFO | Async report generation complete!


Step 10: Async Report (Day 5 - asyncio)


  AI-POWERED SPENDING ANALYSIS REPORT
  Generated: 2026-04-12 17:15:48
------------------------------------------------------------
  Income   : Rs. 61,000.00
  Expenses : Rs. 37,799.00
  Balance  : Rs. 23,201.00

SPENDING ADVICE:
  • [RENT] Rs.15,000.00
    → Rent is a fixed cost. Focus on reducing variable expenses.
  • [FOOD] Rs.6,700.00
    → Consider meal prepping to reduce food expenses by 20-30%!
  • [TRANSPORT] Rs.1,800.00
    → Consider carpooling or public transport to save money.
  • [EDUCATION] Rs.2,999.00
    → Learning is investing in yourself! Great choice.
  • [ENTERTAINMENT] Rs.1,500.00
    → Entertainment spending is high. Try free activities!
  • [HEALTH] Rs.3,500.00
    → Health investment is important - keep it up!
  • [UTILITIES] Rs.2,100.00
    → Check for energy-saving options to reduce utility bills.
  • [SHOPPING] Rs.4,200.00
    → Review subscriptions and one-time purchases carefully.
---------------------------------

In [27]:
# ---- Step 11: Delete demo ----
print("Step 11: Delete Transaction (Day 2 - List operations)")
print()

# Pehli transaction ka ID nikalo
all_txns = list(tracker.transaction_generator())
if all_txns:
    target = all_txns[0]
    print(f"  Deleting: {target}")
    
    result = tracker.delete_transaction(target.transaction_id)
    print(f"  Delete result: {result}")
    print(f"  Transactions remaining: {len(tracker)}")
    print(f"  New balance: Rs. {tracker.balance:,.2f}")

print()
print("=" * 60)
print("  🎉 DEMO COMPLETE!")
print("  All Day 1-5 concepts demonstrated in one project!")
print("=" * 60)

17:15:55 | INFO | Saving 11 transactions to finance_data.json...
17:15:55 | INFO | Data saved successfully to 'finance_data.json'
17:15:55 | INFO | Transaction deleted: TXN-20260412171337754


Step 11: Delete Transaction (Day 2 - List operations)

  Deleting:   [2026-04-12 17:13] SALARY       +Rs.  50000.00  |  Monthly salary - TCS
  Delete result: True
  Transactions remaining: 11
  New balance: Rs. -26,799.00

  🎉 DEMO COMPLETE!
  All Day 1-5 concepts demonstrated in one project!


---
## Step 10 - Interactive Mode (Optional)

Agar aap actual CLI app chalana chahte ho jismein aap input de sako - yeh cell run karo!

> **Note:** Colab mein interactive input limited hai. Yeh locally better kaam karta hai.
> Colab mein `input()` kaam karta hai lekin ek baar mein ek input dena hota hai.


In [29]:
# ============================================================
# INTERACTIVE CLI - Pura Menu System
# ============================================================

async def main_menu():
    """
    Main application loop.
    Day 1 - while loop, if/elif/else
    Day 4 - Error handling
    Day 5 - Async
    """
    print()
    print("=" * 60)
    print("PERSONAL FINANCE TRACKER")
    print("  Python → Gen AI Engineer | Day 6 Project")
    print("=" * 60)
    
    name = input("\n  👤 Enter your name: ").strip() or "User"
    tracker = FinanceTracker(user_name=name)
    
    print(f"\n  Welcome, {name}! Loaded {len(tracker)} existing transactions.")
    
    while True:  # Day 1 - Infinite loop (break se bahar nikalte hain)
        print()
        print_header("MAIN MENU")
        print("  1.Add Transaction")
        print("  2.View Transactions")
        print("  3.Balance Summary")
        print("  4.Category Summary")
        print("  5.Delete Transaction")
        print("  6.Generate AI Report (Async)")
        print("  7.File Info")
        print("  8.Exit")
        print_divider()
        
        choice = input("Choose (1-8): ").strip()
        
        if choice == "1":
            handle_add_transaction(tracker)
        
        elif choice == "2":
            handle_view_transactions(tracker)
        
        elif choice == "3":
            display_balance(tracker)
        
        elif choice == "4":
            display_category_summary(tracker)
        
        elif choice == "5":
            display_all_transactions(tracker)
            txn_id = input("\n  Enter Transaction ID to delete: ").strip()
            if txn_id:
                result = tracker.delete_transaction(txn_id)
                print(f"  {'✅ Deleted!' if result else '❌ Not found.'}")
        
        elif choice == "6":
            await handle_report(tracker)  # Async call!
        
        elif choice == "7":
            info = get_file_info()
            print_header("FILE INFO")
            for k, v in info.items():
                print(f"  {k:20}: {v}")
        
        elif choice == "8":
            print("\nGoodbye! Keep tracking your finances!")
            print("Your data is saved in 'finance_data.json'")
            break  # Loop se bahar niklo
        
        else:
            print("  ❌ Invalid choice. Enter 1-8.")


# ---- Uncomment karke run karo interactive mode ke liye ----
print("Interactive mode ready!")
print("   Neeche wali line ka comment hata kar run karo:")
print("   await main_menu()")
print()
print("   Ya directly run karo:")

# await main_menu()  # ← Is line ka comment hata kar interactive mode chalao!

Interactive mode ready!
   Neeche wali line ka comment hata kar run karo:
   await main_menu()

   Ya directly run karo:


In [ ]:
# ✅ Interactive Mode Run Karne Ke Liye - Is Cell Ko Run Karo
await main_menu()

---
## Step 11 - Concepts Revision Table

Is project mein **kya seekha** - ek nazar mein:


In [31]:
# ============================================================
# CONCEPTS REVISION - Saare concepts ek jagah
# ============================================================

revision = {
    "Day 1 - Basics": [
        "Variables & data types      → transaction fields",
        "if/elif/else               → balance positive/negative check",
        "while loops                → menu loop + input validation",
        "for loops                  → iterate transactions",
        "Functions                  → display_balance(), get_float_input()",
        "F-strings                  → formatted output everywhere",
        "input() / print()          → CLI interface",
    ],
    "Day 2 - Data Structures": [
        "Lists                      → self._transactions storage",
        "Dicts                      → category_summary, report data",
        "Sets                       → categories_used property",
        "List comprehension         → [t for t in ... if ...]",
        "Dict comprehension         → summary building",
        "Set comprehension          → {t.category for t in ...}",
        "sorted() with key          → sorted by amount",
    ],
    "Day 3 - OOP & Pythonic": [
        "@dataclass                 → Transaction class",
        "Class variables            → VALID_CATEGORIES",
        "Instance variables         → self._transactions",
        "__init__                   → FinanceTracker constructor",
        "__str__, __repr__          → print(tracker)",
        "__len__, __iter__          → len(tracker), for t in tracker",
        "@property                  → balance, total_income",
        "@classmethod               → Transaction.from_dict()",
        "Generators (yield)         → transaction_generator()",
        "Inheritance                → Custom exceptions hierarchy",
        "Encapsulation              → _private methods",
        "**kwargs                   → Transaction.from_dict(**data)",
    ],
    "Day 4 - Errors & Files": [
        "Custom exceptions          → InvalidAmountError etc.",
        "try/except/else            → All input handling",
        "Multiple except blocks     → ValueError, PermissionError",
        "raise from                 → FileLoadError from JSONDecodeError",
        "JSON read/write            → save_transactions, load_transactions",
        "Logging                    → logger.info, logger.error",
        "open() with 'w'/'r'        → File operations",
    ],
    "Day 5 - Stdlib & Async": [
        "datetime                   → Transaction timestamps",
        "pathlib.Path               → DATA_FILE path handling",
        "os.path                    → File existence check",
        "asyncio                    → generate_full_report()",
        "async/await                → fetch_spending_analysis()",
        "asyncio.gather()           → Parallel tasks",
        "asyncio.sleep()            → Simulated API delay",
    ]
}

# Print revision table
for day, concepts in revision.items():
    print(f"\n{'='*60}")
    print(f"{day}")
    print(f"{'='*60}")
    for concept in concepts:
        print(f"{concept}")

print(f"\n{'='*60}")
print("Day 6 Project COMPLETE!")
print(f"Total concepts covered: {sum(len(v) for v in revision.values())}")
print(f"{'='*60}")


Day 1 - Basics
Variables & data types      → transaction fields
if/elif/else               → balance positive/negative check
while loops                → menu loop + input validation
for loops                  → iterate transactions
Functions                  → display_balance(), get_float_input()
F-strings                  → formatted output everywhere
input() / print()          → CLI interface

Day 2 - Data Structures
Lists                      → self._transactions storage
Dicts                      → category_summary, report data
Sets                       → categories_used property
List comprehension         → [t for t in ... if ...]
Dict comprehension         → summary building
Set comprehension          → {t.category for t in ...}
sorted() with key          → sorted by amount

Day 3 - OOP & Pythonic
@dataclass                 → Transaction class
Class variables            → VALID_CATEGORIES
Instance variables         → self._transactions
__init__                   → FinanceTrack